In [1]:
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
from datetime import date
from tqdm import tqdm
import time
from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr
from scipy.spatial.distance import jaccard, euclidean, cosine
import functools
import operator

today = date.today()

In [2]:
# 200 bins in rows x 833 biosamples in columns
columns = ['seqname', 'start', 'end']
biosamples = list(range(833))
columns = columns+biosamples

chromatin_state_data = pd.read_csv('../data/All_833_biosamples/matrix.txt.gz', delimiter='\t', names=columns)
# chromatin_state_data_columns = pd.read_csv('../data/All_833_biosamples/matrix.columns.txt', delimiter='|', header=None)

In [3]:
chromatin_state_data

,seqname,start,end,0,1,2,3,4,5,6,...,823,824,825,826,827,828,829,830,831,832
0,chr1,0,200,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
1,chr1,200,400,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
2,chr1,400,600,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
3,chr1,600,800,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
4,chr1,800,1000,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15181503,chrX,155269400,155269600,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
15181504,chrX,155269600,155269800,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
15181505,chrX,155269800,155270000,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
15181506,chrX,155270000,155270200,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18


In [5]:
# chromatin_state_data.to_pickle('../data/epilogos_state_data_18.pkl')
chromatin_state_data = pd.read_pickle('../data/epilogos_state_data_18.pkl')

In [6]:
chromatin_state_data

,seqname,start,end,0,1,2,3,4,5,6,...,823,824,825,826,827,828,829,830,831,832
0,chr1,0,200,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
1,chr1,200,400,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
2,chr1,400,600,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
3,chr1,600,800,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
4,chr1,800,1000,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15181503,chrX,155269400,155269600,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
15181504,chrX,155269600,155269800,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
15181505,chrX,155269800,155270000,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
15181506,chrX,155270000,155270200,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18


In [39]:
indices = list(range(0, len(chromatin_state_data), 1000))
chromatin_state_data_subset = chromatin_state_data.loc[indices]
chromatin_state_data_subset

,seqname,start,end,0,1,2,3,4,5,6,...,823,824,825,826,827,828,829,830,831,832
0,chr1,0,200,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
1000,chr1,200000,200200,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
2000,chr1,400000,400200,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
3000,chr1,600000,600200,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
4000,chr1,800000,800200,18,17,18,17,17,18,18,...,18,18,18,17,18,18,18,18,18,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15177000,chrX,154368800,154369000,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
15178000,chrX,154568800,154569000,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18
15179000,chrX,154768800,154769000,18,13,6,13,18,18,18,...,18,18,18,6,6,18,18,18,18,18
15180000,chrX,154968800,154969000,18,18,18,18,18,18,18,...,18,18,18,18,18,18,18,18,18,18


In [40]:
chromatin_state_data_subset_arr = chromatin_state_data_subset.iloc[:, 3:].to_numpy()

In [41]:
chromatin_state_data_subset.to_pickle('../data/epilogos_state_data_18_1000.pkl')
np.save('../data/epilogos_state_data_18_1000_arr.npy', chromatin_state_data_subset_arr)

In [42]:
temp_arr = np.load('../data/epilogos_state_data_18_500_arr.npy')
temp_arr.shape

(30364, 833)

In [17]:
from sklearn.metrics.pairwise import cosine_similarity
results = cosine_similarity(temp_arr)
print(results)

[[1.        1.        0.9999963 ... 1.        1.        1.       ]
 [1.        1.        0.9999963 ... 1.        1.        1.       ]
 [0.9999963 0.9999963 1.        ... 0.9999963 0.9999963 0.9999963]
 ...
 [1.        1.        0.9999963 ... 1.        1.        1.       ]
 [1.        1.        0.9999963 ... 1.        1.        1.       ]
 [1.        1.        0.9999963 ... 1.        1.        1.       ]]


In [18]:
results.shape

(75908, 75908)

In [28]:
cosine(temp_arr[0], temp_arr[20])
print(sum(temp_arr[0]), sum(temp_arr[20]))

14994 14706


# ARCHS4

In [2]:
import rpy2.robjects as robjects
# import numpy as np

# # load your file
# robjects.r['load']('../data/ARCH4_data/dat_agg.RData')

# # retrieve the matrix that was loaded from the file
# matrix = robjects.r['fname']

# # turn the R matrix into a numpy array
# a = np.array(matrix)

# print(a)

ValueError: r_home is None. Try python -m rpy2.situation

In [43]:
python -m rpy2.situation

SyntaxError: invalid syntax (3377621633.py, line 1)